In [1]:
print("HELLO")

HELLO


In [2]:
import numpy as np
import librosa
import pandas as pd
import os
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score

print("done")

done


In [3]:


data_dir = "Datasets"

feature_dfs = []
transition_dfs = []

for file in os.listdir(data_dir):
    path = os.path.join(data_dir, file)

    if file.endswith("_features.csv"):
        df = pd.read_csv(path)
        df["source"] = file
        feature_dfs.append(df)

    elif file.endswith("_transitions.csv"):
        df = pd.read_csv(path)
        df["source"] = file
        transition_dfs.append(df)

features_df = pd.concat(feature_dfs, ignore_index=True)
transitions_df = pd.concat(transition_dfs, ignore_index=True)

print(features_df.shape)
print(transitions_df.shape)

(124, 20)
(118, 21)


In [ ]:
skrillex_cols = [
    'tempo',
    'rms',
    'centroid',
    'flux',
    'mfcc_0',
    'mfcc_1',
    'mfcc_2',
    'mfcc_3',
    'mfcc_4'
]

for col in skrillex_cols:
    features_df[col] = (
        features_df[col]
        .astype(str)
        .str.strip("[]")
        .astype(float)
    )

In [ ]:
def clean_value(x):
    # Convert "[123.45]" → 123.45
    if isinstance(x, str):
        x = x.strip("[]")
    return float(x)


def get_track_features(track_id, dataframe):
    row = dataframe[dataframe['track_id'] == track_id]

    if len(row) == 0:
        return None

    vals = row.iloc[0][skrillex_cols]

    # Clean every value
    cleaned = [clean_value(v) for v in vals]

    return np.array(cleaned)

In [ ]:
X = []
y = []

#print("Transition columns:", transitions_df.columns.tolist())

# -------------------------------
# Positive transitions (REAL DJ)
# -------------------------------
def add_positive(transitions, features):
    for _, row in transitions.iterrows():
        f1 = get_track_features(row['from_track'], features)
        f2 = get_track_features(row['to_track'], features)

        if f1 is not None and f2 is not None:
            diff = np.abs(f1 - f2)
            X.append(diff)
            y.append(1)

add_positive(transitions_df, features_df)

features2 = pd.read_csv("Datasets/stateOfTrance2026_features.csv")
transitions2= pd.read_csv("Datasets/stateOfTrance2026_transitions.csv")
add_positive(transitions2 , features2)


print("Positive samples:", len(X))


# -------------------------------
# Negative samples (RANDOM)
# -------------------------------
all_tracks = features_df['track_id'].tolist()

transition_pairs = set(zip(
    transitions_df['from_track'],
    transitions_df['to_track']
))

'''
target_size = len(X) * 2

while len(y) < target_size:
    t1, t2 = random.sample(all_tracks, 2)

    if (t1, t2) not in transition_pairs:
        f1 = get_track_features(t1, features_df)
        f2 = get_track_features(t2, features_df)

        if f1 is not None and f2 is not None:
            diff = np.abs(f1 - f2)
            X.append(diff)
            y.append(0)
'''

print("Total samples:", len(X))
print("Positive:", sum(y))
print("Negative:", len(y) - sum(y))

In [ ]:
X = np.array(X)
y = np.array(y)

y = np.array(y).astype(int)
print("Unique labels:", np.unique(y))
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

In [ ]:
model = MLPClassifier(hidden_layer_sizes=(64, 32), alpha=.01, max_iter=300, random_state=42)
model.fit(X_train, y_train)

preds = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, preds))
print(classification_report(y_test, preds))

In [ ]:
audio_folder = "audio"
audio_files = [f for f in os.listdir(audio_folder) if f.endswith(".mp3")]

print(audio_files)

In [ ]:
def get_phrases(y, sr, duration=5, trim_start=10, trim_end=10):
    samples_per_phrase = int(sr * duration)

    start = int(sr * trim_start)
    end = len(y) - int(sr * trim_end)

    y_trimmed = y[start:end]

    phrases = []

    for i in range(0, len(y_trimmed), samples_per_phrase):
        chunk = y_trimmed[i:i+samples_per_phrase]

        if len(chunk) == samples_per_phrase:
            
            energy = np.mean(np.abs(chunk))
            if energy < 0.02:
                continue

            phrases.append(chunk)

    return phrases

In [ ]:
def extract_features(y, sr):
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)

    rms = np.mean(librosa.feature.rms(y=y))
    centroid = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
    flux = np.mean(librosa.onset.onset_strength(y=y, sr=sr))

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=5)
    mfcc_means = np.mean(mfcc, axis=1)

    return np.hstack([tempo, rms, centroid, flux, mfcc_means])

In [ ]:


if os.path.exists("phrase_features.npy") and os.path.exists("phrase_files.npy"):
    print("Loading cached features...")
    
    files = np.load("phrase_files.npy", allow_pickle=True)
    features = np.load("phrase_features.npy")
    
    phrase_features = list(zip(files, features))

else:
    print("Extracting features (first time)...")
    
    phrase_features = []

    for file in audio_files:
        print(f"Processing: {file}")
        
        path = os.path.join(audio_folder, file)
        y, sr = librosa.load(path, sr=22050)
        
        phrases = get_phrases(y, sr)

        for p in phrases:
            feat = extract_features(p, sr)
            phrase_features.append((file, feat))

    # Save
    files = [f for f, _ in phrase_features]
    features = np.array([feat for _, feat in phrase_features])

    np.save("phrase_files.npy", files)
    np.save("phrase_features.npy", features)

    print("Saved phrase features!")

print("Total phrases:", len(phrase_features))

In [ ]:
import random

phrase_pairs = []

num_samples = 5000

indices = list(range(len(phrase_features)))

for _ in range(num_samples):
    i, j = random.sample(indices, 2)

    f1 = phrase_features[i][1]
    f2 = phrase_features[j][1]
    # print(f1[1],f2[1])
    if f1[1] < 0.1 or f2[1] < 0.1:
        continue

    diff = np.abs(f1 - f2)

    phrase_pairs.append((i, j, diff))

In [ ]:
X_phrases = np.array([p[2] for p in phrase_pairs])
X_phrases_scaled = scaler.transform(X_phrases)

predictions = model.predict(X_phrases_scaled)
probs = model.predict_proba(X_phrases_scaled)[:, 1]
print("Min:", np.min(probs))
print("Max:", np.max(probs))
print("Mean:", np.mean(probs))

In [ ]:
results = []

for idx, (i, j, _) in enumerate(phrase_pairs):
    file_i = phrase_features[i][0]
    file_j = phrase_features[j][0]

    if file_i == file_j:
        continue

    score = probs[idx]

    results.append((i, j, score))

results.sort(key=lambda x: x[2], reverse=True)

for i, j, score in results[:10]:
    print(f"{phrase_features[i][0]} → {phrase_features[j][0]} | Score: {score:.3f}")

In [ ]:
def tempo_baseline(X):
    return (X[:,0] < 10).astype(int)

baseline_preds = tempo_baseline(X_test)

print("Baseline Accuracy:", accuracy_score(y_test, baseline_preds))
print("Baseline F1:", f1_score(y_test, baseline_preds))

In [ ]:
print("Example feature:", X[0])
print("Label:", y[0])

In [ ]:
import librosa
import numpy as np
import os
import soundfile as sf

def save_transition(file_a, file_b, audio_folder, output_folder, idx, duration=5):

    path_a = os.path.join(audio_folder, file_a)
    path_b = os.path.join(audio_folder, file_b)

    y_a, sr = librosa.load(path_a, sr=22050)
    y_b, sr = librosa.load(path_b, sr=22050)

    # Tempo match
    tempo_a, _ = librosa.beat.beat_track(y=y_a, sr=sr)
    tempo_b, _ = librosa.beat.beat_track(y=y_b, sr=sr)

    if tempo_b > 0:
        rate = float(tempo_a / tempo_b)
        rate = np.clip(rate, 0.5, 2.0)
        y_b = librosa.effects.time_stretch(y_b, rate=rate)

    # Segments
    end_a = y_a[-int(sr * duration):]
    start_b = y_b[:int(sr * duration)]

    # Crossfade
    fade_len = int(sr * 2)

    fade_out = np.linspace(1, 0, fade_len)
    fade_in = np.linspace(0, 1, fade_len)

    end_a[-fade_len:] *= fade_out
    start_b[:fade_len] *= fade_in

    combined = np.concatenate([
        end_a[:-fade_len],
        end_a[-fade_len:] + start_b[:fade_len],
        start_b[fade_len:]
    ])
    filename = f"transition_{idx}.wav"
    output_path = os.path.join(output_folder, filename)

    sf.write(output_path, combined, sr)

    print(f"Saved: {filename}")
output_folder = "transitions_output"
os.makedirs(output_folder, exist_ok=True)

In [ ]:
for idx, (i, j, score) in enumerate(results[:50]):
    print(score)
    if (score > .98): continue
    file_a = phrase_features[i][0]
    file_b = phrase_features[j][0]

    print(f"{idx+1}. {file_a} → {file_b} | Score: {score:.3f}")
    
    save_transition(file_a, file_b, audio_folder, output_folder, idx+1)

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.hist(probs, bins=50)
plt.title("Distribution of Transition Scores")
plt.xlabel("Predicted Probability")
plt.ylabel("Frequency")
plt.show()

In [ ]:
good_scores = probs[y_test == 1]
bad_scores = probs[y_test == 0]

plt.figure()
plt.hist(good_scores, bins=30, alpha=0.5)
plt.hist(bad_scores, bins=30, alpha=0.5)
plt.title("Good vs Bad Transition Scores")
plt.xlabel("Probability")
plt.ylabel("Count")
plt.legend(["Good", "Bad"])
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

feature_names = skrillex_cols
feature_importance = np.mean(X, axis=0)

plt.figure()
plt.barh(feature_names, feature_importance)
plt.title("Average Feature Differences")
plt.xlabel("Magnitude")
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, preds)

disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.title("Confusion Matrix")
plt.show()

In [ ]:
top_scores = [score for _, _, score in results[:20]]

plt.figure()
plt.plot(top_scores, marker='o')
plt.title("Top Transition Scores")
plt.xlabel("Rank")
plt.ylabel("Score")
plt.show()

In [ ]:
tempo_diffs = X_phrases[:, 0]

plt.figure()
plt.scatter(tempo_diffs, probs, alpha=0.3)
plt.title("Tempo Difference vs Transition Score")
plt.xlabel("Tempo Difference")
plt.ylabel("Predicted Score")
plt.show()

In [ ]:
from IPython.display import Audio, display

def play_phrase(idx):
    file, feat = phrase_features[idx]
    path = os.path.join(audio_folder, file)
    y, sr = librosa.load(path, sr=22050)
    phrases = get_phrases(y, sr)
    
    count = 0
    for f in audio_files:
        path_f = os.path.join(audio_folder, f)
        y_f, sr_f = librosa.load(path_f, sr=22050)
        p_list = get_phrases(y_f, sr_f)
        
        for p in p_list:
            if count == idx:
                print(f"Playing phrase {idx} from {f}")
                display(Audio(p, rate=sr_f))
                return
            count += 1

In [ ]:
play_phrase(0)
play_phrase(10)
play_phrase(50)